## Databricks Homework
Since in July 2025 Databricks Community Edition was deprecated and instead of creating separate cluster they are being provided in serverless mode it will be easier for you to work with data - since all the data and tables will be saving not only when cluster as active.

So, no separate activities for cluser creating should be executed - it will be autoattached/started when you will execute any of the cells below.


Please, create table in the default schema using file Sales_December_2019.csv. On the left found Catalog => Add Data => Drop files to upload, or click to browse => Sales_December_2019.csv After file will be uploaded, just need to confirm that table should be uploaded.

 Make sure that the first row is header selected => Create Table. Table will be created with name that you specified (sales_december_2019 by default) You will be able to change the table name later if needed.

PySpark can process SQL queries as a text. In other words you don't need to switch cell language to SQL.
1. Write data from table that you created into the dataframe using PySpark with SQL query. Show data in the dataframe

In [0]:
# Create DataFrame from SQL query
df = spark.sql("SELECT * FROM workspace.default.sales_december_2019")
df.show()

+--------+--------------------+----------------+----------+--------------+--------------------+
|Order ID|             Product|Quantity Ordered|Price Each|    Order Date|    Purchase Address|
+--------+--------------------+----------------+----------+--------------+--------------------+
|  295665|  Macbook Pro Laptop|               1|      1700|12/30/19 00:01|136 Church St, Ne...|
|  295666|  LG Washing Machine|               1|     600.0|12/29/19 07:03|562 2nd St, New Y...|
|  295667|USB-C Charging Cable|               1|     11.95|12/12/19 18:21|277 Main St, New ...|
|  295668|    27in FHD Monitor|               1|    149.99|12/22/19 15:13|410 6th St, San F...|
|  295669|USB-C Charging Cable|               1|     11.95|12/18/19 12:38|43 Hill St, Atlan...|
|  295670|AA Batteries (4-p...|               1|      3.84|12/31/19 22:58|200 Jefferson St,...|
|  295671|USB-C Charging Cable|               1|     11.95|12/16/19 15:10|928 12th St, Port...|
|  295672|USB-C Charging Cable|         

Any notebook can be parameterized using dbutils.widgets. Try to add one parameter "Product_name" and select data from dataframe filtered by value from this parameter. 

2. Select data where product = "product_name" from dataframe using PySpark

In [0]:
dbutils.widgets.text("Product_name", "USB-C Charging Cable")
product_name = dbutils.widgets.get("Product_name")

df_filtered = df.filter(df["Product"] == product_name)
df_filtered.show()

+--------+--------------------+----------------+----------+--------------+--------------------+
|Order ID|             Product|Quantity Ordered|Price Each|    Order Date|    Purchase Address|
+--------+--------------------+----------------+----------+--------------+--------------------+
|  295667|USB-C Charging Cable|               1|     11.95|12/12/19 18:21|277 Main St, New ...|
|  295669|USB-C Charging Cable|               1|     11.95|12/18/19 12:38|43 Hill St, Atlan...|
|  295671|USB-C Charging Cable|               1|     11.95|12/16/19 15:10|928 12th St, Port...|
|  295672|USB-C Charging Cable|               2|     11.95|12/13/19 09:29|813 Hickory St, D...|
|  295675|USB-C Charging Cable|               2|     11.95|12/13/19 13:52|594 1st St, San F...|
|  295679|USB-C Charging Cable|               1|     11.95|12/25/19 09:39|902 2nd St, Dalla...|
|  295681|USB-C Charging Cable|               1|     11.95|12/25/19 12:37|79 Elm St, Boston...|
|  295682|USB-C Charging Cable|         

As well as in SQL, in PySpark you can use aggregate functions. Package pyspark.sql.functions contains all aggregated function from SQL. Try to perform simple aggregation with dataframe. Don't forget, that column types, which you want to calculate, shoud be numerical.  
3. Calculate the sales for each product, including the number of products sold

In [0]:
from pyspark.sql import functions as F

sales_by_product = (
    df
    .filter(F.col("Quantity Ordered") != "Quantity Ordered")  # Filter out header rows
    .withColumn("Quantity Ordered", F.col("Quantity Ordered").cast("int"))
    .withColumn("Price Each", F.col("Price Each").cast("double"))
    .groupBy("Product")
    .agg(
        F.sum(F.col("Quantity Ordered") * F.col("Price Each")).alias("Total_Sales"),
        F.sum("Quantity Ordered").alias("Total_Quantity_Sold")
    )
    .orderBy(F.col("Total_Sales").desc())
)

sales_by_product.show(20, truncate=False)

+--------------------------+------------------+-------------------+
|Product                   |Total_Sales       |Total_Quantity_Sold|
+--------------------------+------------------+-------------------+
|Macbook Pro Laptop        |1094800.0         |644                |
|iPhone                    |635600.0          |908                |
|ThinkPad Laptop           |540994.5899999965 |541                |
|Google Phone              |429600.0          |716                |
|27in 4K Gaming Monitor    |335781.3899999959 |861                |
|34in Ultrawide Monitor    |322611.50999999605|849                |
|Apple Airpods Headphones  |311850.0          |2079               |
|Flatscreen TV             |199500.0          |665                |
|Bose SoundSport Headphones|182481.74999999825|1825               |
|27in FHD Monitor          |144740.3500000011 |965                |
|Vareebadd Phone           |114000.0          |285                |
|20in Monitor              |62804.28999999966 |5

In the PySpark you can perform dataframe profiling using one of two special commands or simple aggregated functions. Try to find special commands to complete this task or just use aggregated functions. Hint: please, сhange the column data types based on the data in them

4. Show data profiles output for the new dataframe of table sales_december_2019_csv: row count, min and max value for each column

In [0]:
df_typed = (
    df
    .filter(F.col("Quantity Ordered") != "Quantity Ordered")  # Filter out header rows
    .withColumn("Quantity Ordered", F.col("Quantity Ordered").cast("int"))
    .withColumn("Price Each", F.col("Price Each").cast("double"))
    .withColumn("Order Date", F.to_timestamp(F.col("Order Date"), "MM/dd/yy HH:mm"))
)

profiling_sales = df_typed.agg(
    F.count("*").alias("Row_Count"),
    F.count("Quantity Ordered").alias("Qty_count"),
    F.count("Price Each").alias("Price_count"),
    F.count("Order Date").alias("Date_count"),
    F.count("Order ID").alias("ID_count"),
    F.count("Product").alias("Product_count"),
    F.count("Purchase Address").alias("Address_count"),
    F.min("Quantity Ordered").alias("Min_Qty_Ordered"),
    F.max("Quantity Ordered").alias("Max_Qty_Ordered"),
    F.min("Price Each").alias("Min_Price"),
    F.max("Price Each").alias("Max_Price"),
    F.min("Order Date").alias("Min_Date"),
    F.max("Order Date").alias("Max_Date")
)

profiling_sales.show(truncate=False)

+---------+---------+-----------+----------+--------+-------------+-------------+---------------+---------------+---------+---------+-------------------+-------------------+
|Row_Count|Qty_count|Price_count|Date_count|ID_count|Product_count|Address_count|Min_Qty_Ordered|Max_Qty_Ordered|Min_Price|Max_Price|Min_Date           |Max_Date           |
+---------+---------+-----------+----------+--------+-------------+-------------+---------------+---------------+---------+---------+-------------------+-------------------+
|24989    |24989    |24989      |24989     |24989   |24989        |24989        |1              |7              |2.99     |1700.0   |2019-12-01 02:50:00|2020-01-01 05:13:00|
+---------+---------+-----------+----------+--------+-------------+-------------+---------------+---------------+---------+---------+-------------------+-------------------+




5. Add new column to the dataframe from previous task with any default value that you want

In [0]:
df_with_flag = df_typed.withColumn("Data_Source", F.lit("Sales_December_2019.csv"))
df_with_flag.show(5)

+--------+--------------------+----------------+----------+-------------------+--------------------+--------------------+
|Order ID|             Product|Quantity Ordered|Price Each|         Order Date|    Purchase Address|         Data_Source|
+--------+--------------------+----------------+----------+-------------------+--------------------+--------------------+
|  295665|  Macbook Pro Laptop|               1|    1700.0|2019-12-30 00:01:00|136 Church St, Ne...|Sales_December_20...|
|  295666|  LG Washing Machine|               1|     600.0|2019-12-29 07:03:00|562 2nd St, New Y...|Sales_December_20...|
|  295667|USB-C Charging Cable|               1|     11.95|2019-12-12 18:21:00|277 Main St, New ...|Sales_December_20...|
|  295668|    27in FHD Monitor|               1|    149.99|2019-12-22 15:13:00|410 6th St, San F...|Sales_December_20...|
|  295669|USB-C Charging Cable|               1|     11.95|2019-12-18 12:38:00|43 Hill St, Atlan...|Sales_December_20...|
+--------+--------------

Temporary views are processed by cluster and always dropped when the session ends (when the cluster turns off).

6. Create temporary view from task 4 dataframe using PySpark and perform any select using SQL

In [0]:
# code for view creation
df_with_flag.createOrReplaceTempView("sales_december_2019_profiled")

In [0]:
%sql
SELECT Product, COUNT(*) AS Orders_Count, ROUND(AVG(`Price Each`), 2) AS Avg_Price
FROM sales_december_2019_profiled
GROUP BY Product
ORDER BY Orders_Count DESC

Product,Orders_Count,Avg_Price
USB-C Charging Cable,2981,11.95
Lightning Charging Cable,2894,14.95
AAA Batteries (4-pack),2831,2.99
AA Batteries (4-pack),2717,3.84
Wired Headphones,2542,11.99
Apple Airpods Headphones,2059,150.0
Bose SoundSport Headphones,1808,99.99
27in FHD Monitor,962,149.99
iPhone,908,700.0
27in 4K Gaming Monitor,860,389.99
